# Export the MOUSE pipeline for a poster

> **Supplementary poster/testing notebook.** Original processing date: **2026-09-15**; MoDaCor version: **1.8.0**. This is not part of the core MOUSE correction example.

This notebook turns a MoDaCor pipeline YAML into editable graph artwork. It writes:

- the unmodified output of `Pipeline.to_dot()` for provenance;
- a styled Graphviz DOT file with concise poster labels;
- a native, uncompressed `.drawio` document containing editable nodes and connectors;
- an SVG preview of the initial Graphviz layout.

The `.drawio` file is the most convenient starting point for manual spacing, alignment, typography, grouping, and annotation in diagrams.net/draw.io.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import html
import json
import logging
import math
import shlex
import shutil
import subprocess
import sys
import textwrap
import xml.etree.ElementTree as ET

from matplotlib import colormaps, colors
import numpy as np

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "example_utils.py").is_file():
        EXAMPLES_ROOT = candidate
        break
else:
    raise FileNotFoundError("Start Jupyter from MoDaCor_examples or one of its subdirectories.")

sys.path.insert(0, str(EXAMPLES_ROOT))
from example_utils import locate_example_dir
from modacor.runner.pipeline import Pipeline
from modacor.dataclasses.processing_data import ProcessingData
from modacor.io.hdf.hdf_source import HDFSource
from modacor.io.io_sinks import IoSinks
from modacor.io.io_sources import IoSources

PROJECT_DIR = locate_example_dir("BAM/MOUSE")


## Configuration

`TB` is usually a good starting direction for a portrait poster; use `LR` for a wide landscape panel. Node dimensions are specified in Graphviz inches and transferred to draw.io.


In [ ]:
PIPELINE_PATH = PROJECT_DIR / "pipelines" / "MOUSE_solids.yaml"
OUTPUT_DIR = PROJECT_DIR / "work" / "supplementary" / "poster_2026" / "figures"
OUTPUT_STEM = "MOUSE_solids_pipeline_poster"
DIRECTION = "TB"
NODE_WIDTH_IN = 2.30  # minimum; Graphviz expands only when text needs it
NODE_HEIGHT_IN = 0.48
DRAWIO_PIXELS_PER_INCH = 96.0

SHOW_IMPACT_BADGES = True
COMPUTE_IMPACTS = True
IMPACT_SAMPLE_BATCH = 2
IMPACT_CONFIGURATION = 166
RELATIVE_LOG_DEVIATION_QUANTILE = 0.90
IMPACT_COLOR_REFERENCE_QUANTILE = 0.90
IMPACT_COLORMAPS = {"relative": "viridis", "absolute": "plasma"}
BADGE_DIAMETER_PX = 46.0

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_DOT_PATH = OUTPUT_DIR / f"{OUTPUT_STEM}_raw.dot"
POSTER_DOT_PATH = OUTPUT_DIR / f"{OUTPUT_STEM}.dot"
DRAWIO_PATH = OUTPUT_DIR / f"{OUTPUT_STEM}.drawio"
SVG_PATH = OUTPUT_DIR / f"{OUTPUT_STEM}.svg"
IMPACT_JSON_PATH = OUTPUT_DIR / f"{OUTPUT_STEM}_impact_metrics.json"


## Load and inspect the pipeline

Preparing the pipeline validates its dependency graph. `to_spec()` supplies structured node metadata, while `to_dot()` remains the authoritative direct graph export.


In [ ]:
pipeline = Pipeline.from_yaml_file(yaml_file=PIPELINE_PATH)
pipeline.prepare()
spec = pipeline.to_spec()
raw_dot = pipeline.to_dot(direction=DIRECTION)
RAW_DOT_PATH.write_text(raw_dot + "\n", encoding="utf-8")

print(f"Pipeline: {spec['name']}")
print(f"Nodes: {len(spec['nodes'])}; edges: {len(spec['edges'])}")
print(f"Raw MoDaCor DOT: {RAW_DOT_PATH}")


## Optional correction-impact badges

Each selected value-changing correction receives two editable circles at its lower-right corner in the native draw.io file:

- **R** — relative/inter-datapoint impact;
- **A** — absolute-scale impact.

For every valid paired input/output value, $r_i=\ln(|y_{after,i}|/|y_{before,i}|)$. The two displayed magnitudes are:

$$A=100|\operatorname{median}(r_i)|$$

and

$$R=100Q_{0.90}(|r_i-\operatorname{median}(r_i)|).$$

These are symmetric **log-percent** measures: multiplying or dividing by the same factor has the same magnitude. A pure scalar correction has $R=0$. Relative impact uses `viridis`; absolute impact uses `plasma`, and each is normalized independently to its own configurable data quantile. `None` produces a grey placeholder. Sign flips are excluded from the magnitude definition but reported separately as a diagnostic.


In [ ]:
BADGE_STEP_IDS = {
    "TI", "DC", "FF", "FL", "TR",
    "TI_bg", "DC_bg", "FF_bg", "FL_bg", "TR_bg",
    "BG", "SP", "DE", "PO", "SA", "TH",
}
MANUAL_IMPACT_VALUES_LOG_PERCENT = {}  # e.g. {"PO": {"relative": 3.2, "absolute": 8.5}}
SIGNAL_FLOOR_RELATIVE_TO_MEDIAN = 1e-12

def signal_snapshots(processing_data):
    return {
        bundle_key: np.asarray(bundle["signal"].signal).copy()
        for bundle_key, bundle in processing_data.items()
        if "signal" in bundle
    }

def mask_for_bundle(bundle, shape):
    for key in ("mask", "Mask"):
        if key in bundle:
            try:
                return np.broadcast_to(np.asarray(bundle[key].signal, dtype=bool), shape)
            except ValueError:
                pass
    return np.zeros(shape, dtype=bool)

def paired_log_ratios(before, after, mask):
    before = np.asarray(before, dtype=float)
    after = np.asarray(after, dtype=float)
    if before.shape != after.shape:
        return np.empty(0), 0, 0
    finite = np.isfinite(before) & np.isfinite(after) & ~mask
    if not np.any(finite):
        return np.empty(0), 0, 0
    reference = max(
        float(np.median(np.abs(before[finite]))),
        float(np.median(np.abs(after[finite]))),
        np.finfo(float).tiny,
    )
    valid = finite & (np.abs(before) > SIGNAL_FLOOR_RELATIVE_TO_MEDIAN * reference)
    valid &= np.abs(after) > SIGNAL_FLOOR_RELATIVE_TO_MEDIAN * reference
    sign_flips = int(np.count_nonzero(np.signbit(before[valid]) != np.signbit(after[valid])))
    return np.log(np.abs(after[valid]) / np.abs(before[valid])), sign_flips, int(np.count_nonzero(valid))

def impact_metrics(log_ratios, sign_flips, valid_count):
    if log_ratios.size == 0:
        return {"relative": None, "absolute": None, "valid_count": 0, "sign_flip_fraction": None}
    location = float(np.median(log_ratios))
    relative = 100.0 * float(np.quantile(np.abs(log_ratios - location), RELATIVE_LOG_DEVIATION_QUANTILE))
    return {
        "relative": relative, "absolute": 100.0 * abs(location),
        "median_log_percent": 100.0 * location,
        "valid_count": valid_count,
        "sign_flip_fraction": sign_flips / valid_count if valid_count else None,
    }

IMPACT_VALUES_LOG_PERCENT = {
    step_id: {"relative": None, "absolute": None, "median_log_percent": None, "valid_count": 0, "sign_flip_fraction": None}
    for step_id in BADGE_STEP_IDS
}
impact_input = None

if COMPUTE_IMPACTS:
    previous_logging_disable = logging.root.manager.disable
    logging.disable(logging.INFO)
    sys.path.insert(0, str(PROJECT_DIR))
    from mouse_helpers import discover_measurement_pairs

    pair = next(
        candidate for candidate in discover_measurement_pairs(
            PROJECT_DIR / "data", batch_start=IMPACT_SAMPLE_BATCH, batch_end=IMPACT_SAMPLE_BATCH
        )
        if candidate["configuration"] == IMPACT_CONFIGURATION
    )
    impact_input = {
        "sample": pair["sample"].name, "background": pair["background"].name,
        "batch": pair["batch"], "configuration": pair["configuration"],
    }
    impact_sources = IoSources()
    impact_sources.register_source(HDFSource(source_reference="sample", resource_location=pair["sample"]))
    impact_sources.register_source(HDFSource(source_reference="background", resource_location=pair["background"]))
    impact_data = ProcessingData()
    impact_sinks = IoSinks()
    impact_pipeline = Pipeline.from_yaml_file(yaml_file=PIPELINE_PATH)
    scheduler = impact_pipeline.create_scheduler()
    scheduler.prepare()
    finished = False
    while scheduler.is_active() and not finished:
        for node in scheduler.get_ready():
            step_id = str(node.step_id)
            before = signal_snapshots(impact_data) if step_id in BADGE_STEP_IDS else {}
            node.processing_data = impact_data
            node.io_sources = impact_sources
            node.io_sinks = impact_sinks
            node.execute(impact_data)
            if step_id in BADGE_STEP_IDS:
                ratios = []
                sign_flips = valid_count = 0
                for bundle_key, before_signal in before.items():
                    if bundle_key not in impact_data or "signal" not in impact_data[bundle_key]:
                        continue
                    after_signal = np.asarray(impact_data[bundle_key]["signal"].signal)
                    if before_signal.shape != after_signal.shape or np.array_equal(before_signal, after_signal, equal_nan=True):
                        continue
                    mask = mask_for_bundle(impact_data[bundle_key], before_signal.shape)
                    bundle_ratios, bundle_flips, bundle_valid = paired_log_ratios(before_signal, after_signal, mask)
                    if bundle_ratios.size:
                        ratios.append(bundle_ratios)
                        sign_flips += bundle_flips
                        valid_count += bundle_valid
                joined = np.concatenate(ratios) if ratios else np.empty(0)
                IMPACT_VALUES_LOG_PERCENT[step_id] = impact_metrics(joined, sign_flips, valid_count)
            scheduler.done(node)
            if step_id == "TH":
                finished = True
                break
    logging.disable(previous_logging_disable)

for step_id, override in MANUAL_IMPACT_VALUES_LOG_PERCENT.items():
    IMPACT_VALUES_LOG_PERCENT.setdefault(step_id, {}).update(override)

def conventional_display_percent(metric, metrics):
    value = metrics.get(metric)
    if value is None or not math.isfinite(float(value)):
        return None
    if metric == "absolute" and metrics.get("median_log_percent") is not None:
        exponent = float(metrics["median_log_percent"]) / 100.0
        if exponent > math.log(2.0):
            return math.inf
        return 100.0 * abs(math.expm1(exponent))
    exponent = abs(float(value)) / 100.0
    return math.inf if exponent > math.log(2.0) else 100.0 * math.expm1(exponent)

for metrics in IMPACT_VALUES_LOG_PERCENT.values():
    metrics["relative_display_percent"] = conventional_display_percent("relative", metrics)
    metrics["absolute_display_percent"] = conventional_display_percent("absolute", metrics)

def automatic_color_reference(metric, fallback):
    values = [
        float(metrics[metric]) for metrics in IMPACT_VALUES_LOG_PERCENT.values()
        if metrics.get(metric) is not None and math.isfinite(float(metrics[metric])) and float(metrics[metric]) > 0
    ]
    return float(np.quantile(values, IMPACT_COLOR_REFERENCE_QUANTILE)) if values else fallback

IMPACT_COLOR_REFERENCES_LOG_PERCENT = {
    "relative": automatic_color_reference("relative", 10.0),
    "absolute": automatic_color_reference("absolute", 100.0),
}

def impact_color(metric, value_log_percent):
    if value_log_percent is None or not math.isfinite(float(value_log_percent)):
        return "#D9D9D9"
    reference = IMPACT_COLOR_REFERENCES_LOG_PERCENT[metric]
    fraction = abs(float(value_log_percent)) / reference
    return colors.to_hex(colormaps[IMPACT_COLORMAPS[metric]](min(max(fraction, 0.0), 1.0)), keep_alpha=False).upper()

def badge_text_color(fill):
    red, green, blue = colors.to_rgb(fill)
    luminance = 0.2126 * red + 0.7152 * green + 0.0722 * blue
    return "#111111" if luminance > 0.52 else "#FFFFFF"

print(f"Impact badges enabled for {len(BADGE_STEP_IDS)} correction steps.")
print("Independent color references (log-%): " + ", ".join(
    f"{metric}={value:.3g}" for metric, value in IMPACT_COLOR_REFERENCES_LOG_PERCENT.items()
))
for step_id in sorted(BADGE_STEP_IDS):
    metrics = IMPACT_VALUES_LOG_PERCENT[step_id]
    if metrics["relative"] is not None:
        print(
            f"  {step_id}: R={metrics['relative']:.3g}, A={metrics['absolute']:.3g} log-%, "
            f"sign flips={100 * metrics['sign_flip_fraction']:.2f}%"
        )

impact_report = {
    "input": impact_input,
    "units": "log-percent",
    "definitions": {
        "absolute": "100 * abs(median(log(abs(after) / abs(before))))",
        "relative": (
            f"100 * quantile(abs(log-ratio - median(log-ratio)), "
            f"{RELATIVE_LOG_DEVIATION_QUANTILE:g})"
        ),
        "badge_text": "ordinary absolute percentage change; values above 100% are shown as >100%",
    },
    "colormaps": IMPACT_COLORMAPS,
    "color_references_log_percent": IMPACT_COLOR_REFERENCES_LOG_PERCENT,
    "metrics": IMPACT_VALUES_LOG_PERCENT,
}
IMPACT_JSON_PATH.write_text(json.dumps(impact_report, indent=2) + "\n", encoding="utf-8")
print(f"Impact metrics: {IMPACT_JSON_PATH}")


## Make a concise, color-coded DOT version

Colors are organizational hints rather than pipeline semantics. Step IDs remain visible so every poster box can be traced back to the YAML. Edit `stage_for()` or `PALETTE` to establish different visual groups before export.


In [ ]:
PALETTE = {
    "input":      ("#DAE8FC", "#6C8EBF"),
    "mask":       ("#F5F5F5", "#666666"),
    "normalize":  ("#D5E8D4", "#82B366"),
    "combine":    ("#FFE6CC", "#D79B00"),
    "correction": ("#FFF2CC", "#D6B656"),
    "reduction":  ("#E1D5E7", "#9673A6"),
    "output":     ("#F8CECC", "#B85450"),
}

def stage_for(node):
    step_id = str(node["id"])
    module = str(node["module"])
    if module == "AppendProcessingData":
        return "input"
    if "Mask" in module or step_id.startswith(("MK", "CP_")):
        return "mask"
    if step_id in {"BG"} or "SubtractDatabundles" in module:
        return "combine"
    if step_id.startswith(("TI", "DC", "FF", "FL", "TR", "FA")):
        return "normalize"
    if step_id.startswith(("IP", "AV", "CU")) or module in {"IndexPixels", "IndexedAverager"}:
        return "reduction"
    if step_id.startswith(("PL", "DO")) or module.startswith(("Plot", "Sink")):
        return "output"
    return "correction"

def dot_escape(value):
    return str(value).replace("\\", "\\\\").replace('"', '\"')

def compact_title(node):
    title = str(node.get("short_title") or node["module"])
    return "\n".join(textwrap.wrap(title, width=28, break_long_words=False))

def poster_dot_from_spec(pipeline_spec, direction):
    lines = [
        f'digraph "{dot_escape(pipeline_spec["name"])}" {{',
        f"  rankdir={direction};",
        '  graph [bgcolor="transparent", pad="0.12", nodesep="0.18", ranksep="0.20", splines=ortho];',
        f'  node [shape=box, style="rounded,filled", width={NODE_WIDTH_IN}, height={NODE_HEIGHT_IN}, fixedsize=false, margin="0.10,0.05", fontname="Helvetica", fontsize=13, penwidth=1.4];',
        '  edge [color="#666666", arrowsize=0.72, penwidth=1.2];',
    ]
    for node in pipeline_spec["nodes"]:
        fill, stroke = PALETTE[stage_for(node)]
        label = dot_escape(f'{node["id"]}\n{compact_title(node)}')
        lines.append(
            f'  "{dot_escape(node["id"])}" [label="{label}", fillcolor="{fill}", color="{stroke}"];'
        )
    for edge in pipeline_spec["edges"]:
        lines.append(f'  "{dot_escape(edge["from"])}" -> "{dot_escape(edge["to"])}";')
    lines.append("}")
    return "\n".join(lines)

poster_dot = poster_dot_from_spec(spec, DIRECTION)
POSTER_DOT_PATH.write_text(poster_dot + "\n", encoding="utf-8")
print(f"Styled DOT: {POSTER_DOT_PATH}")


## Convert the Graphviz layout to editable draw.io XML

Graphviz determines initial box positions. The converter below transfers those positions, labels, colors, and dependency edges into native mxGraph cells. The file is deliberately uncompressed XML, which keeps it inspectable and version-control friendly.


In [ ]:
def graphviz_plain(dot_source):
    if shutil.which("dot") is None:
        raise RuntimeError("Graphviz 'dot' is required for the initial draw.io layout.")
    completed = subprocess.run(
        ["dot", "-Tplain"], input=dot_source, text=True,
        capture_output=True, check=True,
    )
    return completed.stdout

def parse_plain_layout(plain_text):
    graph_width = graph_height = None
    nodes = {}
    logical_line = ""
    for physical_line in plain_text.splitlines():
        logical_line += ("\n" if logical_line else "") + physical_line
        try:
            fields = shlex.split(logical_line)
        except ValueError:
            # Graphviz plain output may contain literal newlines inside a quoted label.
            continue
        logical_line = ""
        if not fields:
            continue
        if fields[0] == "graph":
            graph_width, graph_height = map(float, fields[2:4])
        elif fields[0] == "node":
            nodes[fields[1]] = {
                "x": float(fields[2]), "y": float(fields[3]),
                "width": float(fields[4]), "height": float(fields[5]),
            }
    if graph_width is None or len(nodes) == 0:
        raise ValueError("Graphviz returned no usable graph layout.")
    return graph_width, graph_height, nodes

def drawio_xml(pipeline_spec, plain_text):
    graph_width, graph_height, layout = parse_plain_layout(plain_text)
    scale = DRAWIO_PIXELS_PER_INCH
    margin = 36.0
    legend_height = 132.0 if SHOW_IMPACT_BADGES else 0.0
    page_width = int(graph_width * scale + 2 * margin)
    page_height = int(graph_height * scale + 2 * margin + legend_height)

    mxfile = ET.Element("mxfile", {
        "host": "app.diagrams.net",
        "modified": datetime.now(timezone.utc).isoformat(),
        "agent": "MoDaCor poster pipeline exporter",
        "version": "26.0.9",
        "compressed": "false",
    })
    diagram = ET.SubElement(mxfile, "diagram", {"id": "modacor-pipeline", "name": "Pipeline"})
    model = ET.SubElement(diagram, "mxGraphModel", {
        "dx": str(page_width), "dy": str(page_height),
        "grid": "1", "gridSize": "10", "guides": "1",
        "tooltips": "1", "connect": "1", "arrows": "1",
        "fold": "1", "page": "1", "pageScale": "1",
        "pageWidth": str(page_width), "pageHeight": str(page_height),
        "math": "1", "shadow": "0",
    })
    root = ET.SubElement(model, "root")
    ET.SubElement(root, "mxCell", {"id": "0"})
    ET.SubElement(root, "mxCell", {"id": "1", "parent": "0"})

    cell_ids = {}
    for index, node in enumerate(pipeline_spec["nodes"], start=2):
        step_id = str(node["id"])
        position = layout[step_id]
        width = position["width"] * scale
        height = position["height"] * scale
        x = margin + position["x"] * scale - width / 2.0
        y = margin + (graph_height - position["y"]) * scale - height / 2.0
        fill, stroke = PALETTE[stage_for(node)]
        cell_id = f"node-{index}"
        cell_ids[step_id] = cell_id
        title_html = html.escape(compact_title(node)).replace("\n", "<br>")
        value = f"<b>{html.escape(step_id)}</b><br>{title_html}"
        style = (
            "rounded=1;whiteSpace=wrap;html=1;arcSize=12;"
            f"fillColor={fill};strokeColor={stroke};strokeWidth=1.4;"
            "fontFamily=Helvetica;fontSize=13;align=center;verticalAlign=middle;"
        )
        cell = ET.SubElement(root, "mxCell", {
            "id": cell_id, "value": value, "style": style,
            "vertex": "1", "parent": "1",
        })
        ET.SubElement(cell, "mxGeometry", {
            "x": f"{x:.2f}", "y": f"{y:.2f}",
            "width": f"{width:.2f}", "height": f"{height:.2f}",
            "as": "geometry",
        })

        if SHOW_IMPACT_BADGES and step_id in BADGE_STEP_IDS:
            values = IMPACT_VALUES_LOG_PERCENT.get(step_id, {})
            # Child coordinates are local to the process box, so badges move with it in draw.io.
            badge_y = height - BADGE_DIAMETER_PX / 2.0
            for badge_index, (metric, letter) in enumerate((("relative", "R"), ("absolute", "A"))):
                value_log_percent = values.get(metric)
                fill_badge = impact_color(metric, value_log_percent)
                font_badge = badge_text_color(fill_badge)
                badge_x = width - (2 - badge_index) * BADGE_DIAMETER_PX - (1 - badge_index) * 4.0 + 3.0
                display_percent = values.get(f"{metric}_display_percent")
                if display_percent is None:
                    display_label = "—"
                elif not math.isfinite(display_percent) or display_percent > 100.0:
                    display_label = ">100%"
                else:
                    display_label = f"{display_percent:.0f}%"
                tooltip_value = (
                    "not quantified" if value_log_percent is None
                    else f"{value_log_percent:.4g} log-%, displayed as {display_label}"
                )
                badge = ET.SubElement(root, "mxCell", {
                    "id": f"badge-{step_id}-{metric}",
                    "value": f"<b>{letter}</b><br>{display_label}",
                    "tooltip": f"{metric} impact: {tooltip_value}",
                    "impactMetric": metric,
                    "impactLogPercent": "" if value_log_percent is None else str(value_log_percent),
                    "signFlipFraction": "" if values.get("sign_flip_fraction") is None else str(values["sign_flip_fraction"]),
                    "style": (
                        "ellipse;whiteSpace=wrap;html=1;aspect=fixed;"
                        f"fillColor={fill_badge};strokeColor=#303030;strokeWidth=1.2;"
                        f"fontColor={font_badge};fontSize=9;align=center;verticalAlign=middle;"
                    ),
                    "vertex": "1", "parent": cell_id,
                })
                ET.SubElement(badge, "mxGeometry", {
                    "x": f"{badge_x:.2f}", "y": f"{badge_y:.2f}",
                    "width": f"{BADGE_DIAMETER_PX:.2f}", "height": f"{BADGE_DIAMETER_PX:.2f}",
                    "as": "geometry",
                })

    edge_style = (
        "edgeStyle=orthogonalEdgeStyle;rounded=0;orthogonalLoop=1;"
        "jettySize=auto;html=1;strokeColor=#666666;strokeWidth=1.2;"
        "endArrow=block;endFill=1;"
    )
    for index, edge in enumerate(pipeline_spec["edges"], start=1):
        cell = ET.SubElement(root, "mxCell", {
            "id": f"edge-{index}", "style": edge_style, "edge": "1",
            "parent": "1", "source": cell_ids[str(edge["from"])],
            "target": cell_ids[str(edge["to"])],
        })
        ET.SubElement(cell, "mxGeometry", {"relative": "1", "as": "geometry"})

    if SHOW_IMPACT_BADGES:
        legend_y = margin + graph_height * scale + 18.0
        legend_value = (
            "<b>Impact badges</b> — R: relative/inter-datapoint change · "
            "A: absolute-scale change · circle text: rounded ordinary |Δ|% · "
            "grey: not quantified · colors: independently scaled symmetric log-%"
        )
        legend = ET.SubElement(root, "mxCell", {
            "id": "impact-legend", "value": legend_value,
            "style": "text;html=1;strokeColor=none;fillColor=none;align=left;verticalAlign=middle;fontSize=12;",
            "vertex": "1", "parent": "1",
        })
        ET.SubElement(legend, "mxGeometry", {
            "x": f"{margin:.2f}", "y": f"{legend_y:.2f}",
            "width": f"{page_width - 2 * margin:.2f}", "height": "28.00", "as": "geometry",
        })
        for metric_index, (metric, letter) in enumerate((("relative", "R"), ("absolute", "A"))):
            scale_y = legend_y + 36.0 + metric_index * 32.0
            reference = IMPACT_COLOR_REFERENCES_LOG_PERCENT[metric]
            scale_title = f"{letter} · {IMPACT_COLORMAPS[metric]} · log-%"
            title = ET.SubElement(root, "mxCell", {
                "id": f"impact-scale-title-{metric}", "value": scale_title,
                "style": "text;html=1;strokeColor=none;fillColor=none;align=left;verticalAlign=middle;fontStyle=1;fontSize=10;",
                "vertex": "1", "parent": "1",
            })
            ET.SubElement(title, "mxGeometry", {
                "x": f"{margin:.2f}", "y": f"{scale_y - 1.0:.2f}",
                "width": "145.00", "height": "20.00", "as": "geometry",
            })
            for scale_index, fraction in enumerate((0.0, 0.25, 0.50, 0.75, 1.0)):
                value_log_percent = fraction * reference
                circle_x = margin + 150.0 + scale_index * 86.0
                fill_badge = impact_color(metric, value_log_percent)
                circle = ET.SubElement(root, "mxCell", {
                    "id": f"impact-scale-{metric}-{scale_index}", "value": "",
                    "style": f"ellipse;aspect=fixed;fillColor={fill_badge};strokeColor=#303030;strokeWidth=1;",
                    "vertex": "1", "parent": "1",
                })
                ET.SubElement(circle, "mxGeometry", {
                    "x": f"{circle_x:.2f}", "y": f"{scale_y:.2f}",
                    "width": "18.00", "height": "18.00", "as": "geometry",
                })
                label = ET.SubElement(root, "mxCell", {
                    "id": f"impact-scale-label-{metric}-{scale_index}", "value": f"{value_log_percent:.3g}",
                    "style": "text;html=1;strokeColor=none;fillColor=none;align=left;verticalAlign=middle;fontSize=10;",
                    "vertex": "1", "parent": "1",
                })
                ET.SubElement(label, "mxGeometry", {
                    "x": f"{circle_x + 22.0:.2f}", "y": f"{scale_y - 1.0:.2f}",
                    "width": "50.00", "height": "20.00", "as": "geometry",
                })

    tree = ET.ElementTree(mxfile)
    ET.indent(tree, space="  ")
    return ET.tostring(mxfile, encoding="unicode") + "\n"

plain_layout = graphviz_plain(poster_dot)
DRAWIO_PATH.write_text(drawio_xml(spec, plain_layout), encoding="utf-8")
subprocess.run(["dot", "-Tsvg", "-o", str(SVG_PATH), str(POSTER_DOT_PATH)], check=True)
print(f"Editable draw.io: {DRAWIO_PATH}")
print(f"SVG preview: {SVG_PATH}")


## Validate and open

Open the `.drawio` file directly in draw.io. Every process step is an independent rounded rectangle and every dependency is an independent connector. The styled `.dot` can alternatively be inserted through **Arrange → Insert → Advanced → Text → Graphviz** in draw.io versions that provide that importer.


In [ ]:
drawio_root = ET.parse(DRAWIO_PATH).getroot()
drawio_vertices = drawio_root.findall(".//mxCell[@vertex='1']")
drawio_edges = drawio_root.findall(".//mxCell[@edge='1']")
drawio_badges = [cell for cell in drawio_vertices if cell.attrib.get("impactMetric")]
assert len(drawio_vertices) >= len(spec["nodes"])
assert len(drawio_edges) == len(spec["edges"])
assert len(drawio_badges) == (2 * len(BADGE_STEP_IDS) if SHOW_IMPACT_BADGES else 0)

manifest = {
    "source_pipeline": str(PIPELINE_PATH.relative_to(PROJECT_DIR)),
    "pipeline_name": spec["name"],
    "node_count": len(spec["nodes"]),
    "edge_count": len(spec["edges"]),
    "impact_badge_count": len(drawio_badges),
    "direction": DIRECTION,
    "outputs": [str(path.relative_to(PROJECT_DIR)) for path in (RAW_DOT_PATH, POSTER_DOT_PATH, DRAWIO_PATH, SVG_PATH, IMPACT_JSON_PATH)],
}
print(json.dumps(manifest, indent=2))
